# A3T-GCN ablation (E0 / E1 / E2) on Kaggle

E0 = A3T-GCN, E1 = A3T-GCN + VAR, E2 = A3T-GCN + VAR + PEM. Only the graph differs; all
hyperparameters are fixed in `config.py` of the cloned repository and verified in section 2b.

**Before running**
1. *Add Input* → the GBA dataset (`gba_his_raw_2021.h5`, `gba_rn_adj.npy`, `gba_meta.csv`).
2. *Add Input* → the artifacts dataset (`E1__*.npz`, `E2__*.npz`, `sensor_filter_*.json`); needed for E1 / E2.
3. Settings: **Accelerator = GPU P100**, **Internet = On**, and keep the **same environment** for E0, E1, E2.
4. Change only `EXPERIMENT` in the settings cell, then **Save Version → Save & Run All (Commit)**.
   Only commit runs keep their files: download `output_<EXPERIMENT>_final` from that version's *Output* tab.


## Run settings (change only EXPERIMENT)

In [ ]:
# Final runs: change ONLY the experiment. Everything else follows from it.
EXPERIMENT     = "E1"      # "E0" (A3T-GCN), "E1" (+VAR), "E2" (+VAR+PEM)

FINAL_GRAPHS = {
    "E0": None,
    "E1": "E1__lag3_alpha1e+06.npz",
    "E2": "E2__lag3_alpha1e+06__nu0_0.0001_nu1_0.002_tau_0.01_thr_0.5_eta_0.5_L-BFGS-B.npz",
}
GRAPH_FILE     = FINAL_GRAPHS[EXPERIMENT]
RUN_NAME       = f"{EXPERIMENT}_final"

# Leave as they are for the final runs
EPOCHS         = None      # None = 20 epochs from config.py
SELECTION_ONLY = False     # False = the test set is evaluated once at the end

REPO_URL       = "https://github.com/prof-sanjay/A3TGCN-VAR-PEM-Implementation.git"
REPO_BRANCH    = "main"

# Data and artifacts are found automatically under /kaggle/input (section 2).
print("Experiment:", EXPERIMENT, "| graph:", GRAPH_FILE, "| run name:", RUN_NAME)


## 1. Environment (Kaggle's built-in TensorFlow; note the versions - keep them identical for E0, E1, E2)

In [ ]:
import sys
print(sys.version)
print(sys.version_info)

In [ ]:
import sys
import tensorflow as tf
import numpy as np
import scipy
import h5py
import pandas as pd

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("h5py:", h5py.__version__)
print("Pandas:", pd.__version__)

print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Data, code (fresh clone) and artifacts

In [ ]:
import os
import glob
import shutil
import subprocess

# Folder under /kaggle/input that contains the LargeST-GBA files
# (e.g. /kaggle/input/datasets/sanjumurali/gba-dataset).
hits = glob.glob("/kaggle/input/**/gba_his_raw_2021.h5", recursive=True)
assert hits, "gba_his_raw_2021.h5 not found under /kaggle/input - add the GBA dataset (Add Input)"
DATA_DIR = os.path.dirname(hits[0])

print("DATA_DIR:", DATA_DIR)

required_files = [
    "gba_his_raw_2021.h5",
    "gba_rn_adj.npy",
    "gba_meta.csv",
]

for name in required_files:
    path = os.path.join(DATA_DIR, name)
    assert os.path.exists(path), f"Missing dataset file: {path}"
    print("✓", path)


In [ ]:
WORK = "/kaggle/working"
CODE = f"{WORK}/implementation"  # package name must be 'implementation'

# ------------------------------------------------------------------
# Clone the implementation repository
# ------------------------------------------------------------------
if os.path.exists(CODE):
    shutil.rmtree(CODE)

subprocess.run(
    ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, CODE],
    check=True
)

print(
    subprocess.run(
        ["git", "-C", CODE, "log", "-1", "--oneline"],
        capture_output=True,
        text=True
    ).stdout
)

# ------------------------------------------------------------------
# Dataset: link DATA_DIR (previous cell) to the location expected by
# the implementation: implementation/datasets/LargeST GBA/
# ------------------------------------------------------------------
DATA_TARGET = f"{CODE}/datasets/LargeST GBA"

os.makedirs(f"{CODE}/datasets", exist_ok=True)

if os.path.lexists(DATA_TARGET):
    if os.path.islink(DATA_TARGET):
        os.unlink(DATA_TARGET)
    else:
        shutil.rmtree(DATA_TARGET)

os.symlink(DATA_DIR, DATA_TARGET)

for name in required_files:
    path = os.path.join(DATA_TARGET, name)
    assert os.path.exists(path), f"Missing dataset file: {path}"

print("Dataset linked successfully:")
for name in required_files:
    print("  ✓", name)

# ------------------------------------------------------------------
# Artifacts: E1 / E2 graphs and the sensor-filter cache, found
# anywhere under /kaggle/input (no fixed dataset path needed)
# ------------------------------------------------------------------
os.makedirs(f"{CODE}/outputs/graphs", exist_ok=True)
os.makedirs(f"{CODE}/outputs/sensor_filter", exist_ok=True)

for path in glob.glob("/kaggle/input/**/*", recursive=True):
    if not os.path.isfile(path):
        continue
    filename = os.path.basename(path)
    if filename.startswith(("E1__", "E2__")) and filename.endswith((".npz", ".json")):
        # Kaggle drops "+" from uploaded file names (alpha1e+06 -> alpha1e06);
        # restore the original name expected by config.py.
        original = filename.replace("alpha1e06", "alpha1e+06")
        shutil.copy(path, f"{CODE}/outputs/graphs/{original}")
    elif filename.startswith("sensor_filter_") and filename.endswith(".json"):
        shutil.copy(path, f"{CODE}/outputs/sensor_filter/")

print("graphs:", sorted(os.listdir(f"{CODE}/outputs/graphs")))
print("sensor filter cache:", os.listdir(f"{CODE}/outputs/sensor_filter"))

assert GRAPH_FILE is None or os.path.exists(f"{CODE}/outputs/graphs/{GRAPH_FILE}"), \
    f"{GRAPH_FILE} not found - add the artifacts dataset (Add Input)"


## 2b. Fixed hyperparameters (identical for E0, E1, E2): verified, do not edit

In [ ]:
# Fixed hyperparameters of the ablation: identical for E0, E1 and E2.
# The run stops here if the cloned code does not contain exactly these values.
EXPECTED = {
    # shared A3T-GCN / training settings
    "learning_rate": 0.001,
    "grad_clip_norm": 5.0,
    "training_epoch": 20,
    "batch_size": 8,
    "gru_units": 100,
    "seq_len": 12,
    "pre_len": 1,
    "lambda_loss": 0.0015,
    "lr_patience": 5,
    "lr_factor": 0.5,
    "train_rate": 0.8,
    "val_rate": 0.1,
    "seed": 42,
    "sensor_filter": "duplicates",
    "imputation": "gap_bands",
    # component settings (E1 / E2)
    "VAR_LAG": 3,
    "VAR_ALPHA": 1e6,
    "PEM_NU0": 1e-4,
    "PEM_NU1": 2e-3,
}

import importlib
import sys
sys.path.insert(0, WORK)
cfg_module = importlib.import_module("implementation.config.config")
cfg = cfg_module.ExperimentConfig(experiment=EXPERIMENT)

actual = {k: getattr(cfg, k) if hasattr(cfg, k) else getattr(cfg_module, k) for k in EXPECTED}
mismatch = {k: (actual[k], EXPECTED[k]) for k in EXPECTED if actual[k] != EXPECTED[k]}

for k, v in actual.items():
    print(f"{k:16s} {v}")
if EPOCHS is not None:
    print(f"NOTE: EPOCHS override = {EPOCHS} (final runs must use None)")

assert not mismatch, f"Cloned code has different hyperparameters (actual, expected): {mismatch}. Push the latest code."
cfg.validate()
print("\nFixed hyperparameters verified for", EXPERIMENT, "| graph:", cfg.graph_path)


## 3. Run

In [ ]:
cmd = [sys.executable, "-u", "-m", "implementation.main", "--experiment", EXPERIMENT]
if GRAPH_FILE:
    cmd += ["--graph", f"{CODE}/outputs/graphs/{GRAPH_FILE}"]
if EPOCHS is not None:
    cmd += ["--epochs", str(EPOCHS)]
if SELECTION_ONLY:
    cmd += ["--selection-only"]
if RUN_NAME:
    cmd += ["--run-name", RUN_NAME]
print(" ".join(cmd))

env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="2", PYTHONUNBUFFERED="1")
with subprocess.Popen(cmd, cwd=WORK, env=env, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:
        if not any(noise in line for noise in ("WARNING:tensorflow", "Instructions for updating", "non-resource variables")):
            print(line, end="")
assert process.returncode == 0, f"main.py exited with {process.returncode}"

## 4. Results (saved as notebook output)

In [ ]:
import json, os, shutil

# Uses EXPERIMENT / RUN_NAME from the settings cell.
run_dir = f"{CODE}/results/{RUN_NAME or EXPERIMENT}"
print("Looking in:", run_dir, "->", os.path.isdir(run_dir))
assert os.path.isdir(run_dir), "Run folder not found - did section 3 finish without errors?"

print("Files:", sorted(os.listdir(run_dir)))
results = json.load(open(f"{run_dir}/results.json"))
summary = {k: results.get(k) for k in ["experiment", "best_epoch", "best_val_rmse_original",
                                         "training_seconds", "valid_sequences", "test_metrics"]}
summary["learning_rate"] = results["config"].get("learning_rate")
summary["grad_clip_norm"] = results["config"].get("grad_clip_norm")
summary["graph"] = results.get("graph", {}).get("source")
print(json.dumps(summary, indent=2))

out = f"{WORK}/output_{RUN_NAME or EXPERIMENT}"
shutil.copytree(run_dir, out, dirs_exist_ok=True)
print("Saved to", out, "->", sorted(os.listdir(out)))

if not SELECTION_ONLY:
    assert os.path.exists(f"{out}/test_predictions.npz"), "test_predictions.npz missing!"
print("Download this folder from the version's Output tab:", out)


## 5. Visualization of this run

Training curve; for final runs also test metrics, MAE by time of day, per-sensor MAE and observed vs predicted. Figures are saved into the output folder.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
try:
    from IPython.display import display
except ImportError:          # plain Python (outside a notebook)
    display = print

BLUE, INK, MUTED, GRID, SURFACE = "#2a78d6", "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "axes.titlecolor": INK,
    "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False,
})

label = RUN_NAME or EXPERIMENT
history = pd.read_csv(f"{out}/history.csv") if os.path.exists(f"{out}/history.csv") \
    else pd.DataFrame(results["history"])

# 1. Training curve: validation RMSE / MAE (top), learning rate (bottom, own axis)
fig, (ax, ax_lr) = plt.subplots(2, 1, figsize=(7.5, 5), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})
ax.plot(history["epoch"], history["val_rmse_original"], color=BLUE, marker="o", ms=4, label="Val RMSE")
ax.plot(history["epoch"], history["val_mae_original"], color=BLUE, ls="--", marker="o", ms=3, label="Val MAE")
best = history["val_rmse_original"].idxmin()
ax.plot(history["epoch"][best], history["val_rmse_original"][best], "o", ms=10,
        mfc=SURFACE, mec=BLUE, mew=2, label=f"best epoch {int(history['epoch'][best])}")
ax.set_ylabel("Vehicles / 5 min")
ax.set_title(f"{label}: validation error per epoch")
ax.legend()
ax_lr.step(history["epoch"], history["learning_rate"], where="post", color=MUTED)
ax_lr.set_ylabel("LR")
ax_lr.set_xlabel("Epoch")
ax_lr.xaxis.set_major_locator(MaxNLocator(integer=True))
fig.tight_layout(); fig.savefig(f"{out}/fig_training_curve.png", dpi=200); plt.show()

if not SELECTION_ONLY and os.path.exists(f"{out}/test_predictions.npz"):

    # 2. Test metrics table
    metrics = {k: v for k, v in results["test_metrics"].items() if k != "mape_min_flow"}
    table = pd.DataFrame([metrics], index=[label])
    table["best_epoch"] = results["best_epoch"]
    table["best_val_rmse"] = results["best_val_rmse_original"]
    table["training_h"] = results["training_seconds"] / 3600
    display(table.round(4))
    display(pd.DataFrame(results["test_breakdown"]).T.round(4))

    pred = np.load(f"{out}/test_predictions.npz")
    per_sensor = pd.read_csv(f"{out}/per_sensor_metrics.csv")
    tod = pd.read_csv(f"{out}/time_of_day_metrics.csv")

    # 3. MAE by time of day and per-sensor MAE distribution
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
    a1.plot(tod["slot"] * 5 / 60, tod["mae"].rolling(3, center=True, min_periods=1).mean(), color=BLUE)
    a1.set_xlim(0, 24); a1.set_xticks(range(0, 25, 3))
    a1.set_xlabel("Time of day (h)"); a1.set_ylabel("Test MAE (veh / 5 min)")
    a1.set_title("Test MAE by time of day")
    a2.hist(per_sensor["mae"], bins=50, color=BLUE, edgecolor=SURFACE, linewidth=0.5)
    a2.axvline(per_sensor["mae"].median(), color=INK, ls="--", lw=1,
               label=f"median {per_sensor['mae'].median():.1f}")
    a2.set_xlabel("Per-sensor test MAE"); a2.set_ylabel("Sensors")
    a2.set_title(f"Per-sensor MAE ({len(per_sensor)} sensors)"); a2.legend()
    fig.tight_layout(); fig.savefig(f"{out}/fig_test_errors.png", dpi=200); plt.show()

    # 4. Observed vs predicted, first 3 test days, median- and high-flow sensor
    times = pd.to_datetime(pred["timestamps_ns"])
    window = times < times[0] + pd.Timedelta(days=3)
    order = np.argsort(pred["targets"].mean(axis=0))
    picks = [order[len(order) // 2], order[int(0.9 * (len(order) - 1))]]
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    for a, j in zip(axes, picks):
        a.plot(times[window], pred["targets"][window, j], color=INK, lw=1.1, label="Observed")
        a.plot(times[window], pred["predictions"][window, j], color=BLUE, lw=1.0, label=label)
        a.set_title(f"Sensor {int(pred['sensor_ids'][j])}"); a.set_ylabel("Flow"); a.legend(loc="upper right")
    fig.tight_layout(); fig.savefig(f"{out}/fig_example_series.png", dpi=200); plt.show()

print("Figures saved in", out, "->", [f for f in os.listdir(out) if f.startswith("fig_")])
